In [15]:
import pandas as pd

import numpy as np


from pathlib import Path
 



In [16]:
# -----------------------------

# 1) LOAD + COMBINE ALL CSVs

# -----------------------------

data_dir = Path("../Raw Data")
 
csv_files = sorted([*data_dir.glob("*.csv"), *data_dir.glob("*.CSV")])

if not csv_files:

    raise FileNotFoundError(f"No CSV files found in {data_dir}. Check the folder path.")
 
df_raw = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)
 
# Optional: track file source (useful for seasonal patterns)

# If you want this, uncomment below and re-load in a loop instead of concat generator.

# dfs=[]

# for f in csv_files:

#     tmp = pd.read_csv(f)

#     tmp["source_file"] = f.name

#     dfs.append(tmp)

# df_raw = pd.concat(dfs, ignore_index=True)
 
print("Raw shape:", df_raw.shape)

print("Columns:", list(df_raw.columns))
 

Raw shape: (3730, 25)
Columns: ['case_id', 'snapshot_at', 'created_at', 'channel', 'case_type', 'category', 'subcategory', 'priority', 'sla_target_hours', 'first_response_time_hours', 'resolution_time_hours', 'status', 'resolution_code', 'escalated', 'assigned_team', 'escalation_team', 'customer_tenure_months', 'plan_tier', 'region_uk', 'age_band', 'gender', 'case_summary', 'sentiment', 'csat_score', 'tags']


In [17]:
# -----------------------------

# 2) BASIC TYPE CLEANING (DATES)

# -----------------------------

# Convert date-like strings to datetime (errors='coerce' -> invalid becomes NaT)

for col in ["snapshot_at", "created_at"]:

    if col in df_raw.columns:

        df_raw[col] = pd.to_datetime(df_raw[col], errors="coerce")

In [18]:
# -----------------------------

# 3) DQ: CLEAN 'gender' -> 'gender_clean'

#    (Male, Female, Non-Binary only; everything else -> NA)

# -----------------------------

if "gender" in df_raw.columns:

    g_norm = (

        df_raw["gender"]

        .astype("string")

        .str.strip()

        .str.lower()

    )
 
    df_raw["gender_clean"] = np.select(

        [

            g_norm.isin(["m", "male"]),

            g_norm.isin(["f", "female"]),

            g_norm.isin(["nb", "non-binary", "nonbinary", "non binary"]),

        ],

        ["Male", "Female", "Non-Binary"],

        default=pd.NA

    )
 
print("\nGender_clean counts:")

print(df_raw["gender_clean"].value_counts(dropna=False))
 


Gender_clean counts:
gender_clean
Male          1894
Female        1750
<NA>            54
Non-Binary      32
Name: count, dtype: int64


In [19]:

# -----------------------------

# 4) FILTER OUT BLANK age_band

#    (removes NaN, empty string, whitespace-only)

# -----------------------------

if "age_band" in df_raw.columns:

    df_clean = df_raw[

        df_raw["age_band"].notna() &

        df_raw["age_band"].astype(str).str.strip().ne("")

    ].copy()

else:

    df_clean = df_raw.copy()
 
print("\nAfter filtering blank age_band:", df_clean.shape)
 
 


After filtering blank age_band: (3676, 26)


In [20]:
# -----------------------------

# 5) REMOVE DUPLICATE ROWS

# -----------------------------

before = df_clean.shape[0]

df_clean = df_clean.drop_duplicates().copy()

after = df_clean.shape[0]

print(f"\nDropped {before - after} duplicate rows. New shape: {df_clean.shape}")
 
 


Dropped 1973 duplicate rows. New shape: (1703, 26)


In [21]:
# -----------------------------

# 6) (OPTIONAL) QUICK DQ SUMMARY

# -----------------------------

dq_summary = pd.DataFrame({

    "dtype": df_clean.dtypes,

    "null_count": df_clean.isna().sum(),

    "null_pct": (df_clean.isna().mean() * 100).round(2),

    "distinct_count": df_clean.nunique()

}).sort_values("null_pct", ascending=False)
 
print("\nTop null% columns:")

print(dq_summary.head(10))
 
 


Top null% columns:
                                         dtype  null_count  null_pct  \
escalation_team                         object        1535     90.14   
csat_score                              object         720     42.28   
resolution_code                         object         491     28.83   
resolution_time_hours                  float64         460     27.01   
first_response_time_hours              float64           1      0.06   
channel                                 object           0      0.00   
snapshot_at                datetime64[ns, UTC]           0      0.00   
created_at                 datetime64[ns, UTC]           0      0.00   
case_type                               object           0      0.00   
case_id                                 object           0      0.00   

                           distinct_count  
escalation_team                         6  
csat_score                             10  
resolution_code                         7  
resolution_

In [25]:
# Save raw data for model to call as csv
path = r"C:\Users\barnyrumbold\OneDrive - Kidney Research UK\Desktop\Hackathon\Northstar-Desk-Decision-Support-Tool-\Data\clean.csv"
df_clean.to_csv(path, index=False, encoding='utf-8')

In [ ]:
# -----------------------------

# 7) MODEL SETUP: 'priority' AS TARGET

#    (Leakage-safe feature set + text)

# -----------------------------

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, confusion_matrix
 
# Ensure target exists and is clean

if "priority" not in df_clean.columns:

    raise KeyError("Column 'priority' not found in dataset.")
 
# Choose features that are typically available at intake time (avoid leakage)

text_col = "case_summary"
 
cat_cols = [

    "channel", "case_type", "category", "subcategory",

    "assigned_team", "plan_tier", "region_uk",

    "age_band", "gender_clean", "sentiment", "tags"

]
 
num_cols = ["customer_tenure_months"]
 
# Keep only columns that actually exist (defensive programming)

cat_cols = [c for c in cat_cols if c in df_clean.columns]

num_cols = [c for c in num_cols if c in df_clean.columns]
 
# Build X / y

X = df_clean[cat_cols + num_cols + ([text_col] if text_col in df_clean.columns else [])].copy()

y = df_clean["priority"].copy()
 
# Fill missing text with empty string (TF-IDF can't handle NaN)

if text_col in X.columns:

    X[text_col] = X[text_col].fillna("")
 
# Train/test split (stratify to keep class balance)

X_train, X_test, y_train, y_test = train_test_split(

    X, y, test_size=0.2, random_state=42, stratify=y

)
 
# Preprocessing blocks

transformers = []
 
if text_col in X.columns:

    transformers.append(

        ("text", TfidfVectorizer(ngram_range=(1, 2), min_df=2), text_col)

    )
 
if cat_cols:

    transformers.append(

        ("cat", Pipeline(steps=[

            ("imputer", SimpleImputer(strategy="most_frequent")),

            ("onehot", OneHotEncoder(handle_unknown="ignore"))

        ]), cat_cols)

    )
 
if num_cols:

    transformers.append(

        ("num", Pipeline(steps=[

            ("imputer", SimpleImputer(strategy="median"))

        ]), num_cols)

    )
 
preprocess = ColumnTransformer(transformers=transformers, remainder="drop")
 
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
 
model = Pipeline(steps=[

    ("preprocess", preprocess),

    ("clf", clf)

])
 
# Train

model.fit(X_train, y_train)
 
# Evaluate

pred = model.predict(X_test)
 
print("\nClassification report:")

print(classification_report(y_test, pred))
 
print("\nConfusion matrix:")

print(confusion_matrix(y_test, pred))
 
# If you want probabilities (confidence):

if hasattr(model.named_steps["clf"], "predict_proba"):

    proba = model.predict_proba(X_test.iloc[:5])

    print("\nExample predicted probabilities (first 5 rows):")

    print(pd.DataFrame(proba, columns=model.named_steps["clf"].classes_))

 


Classification report:
              precision    recall  f1-score   support

        High       0.66      0.78      0.72        74
         Low       0.73      0.89      0.80        81
      Medium       0.88      0.64      0.74       146
      Urgent       0.70      0.82      0.76        40

    accuracy                           0.75       341
   macro avg       0.74      0.79      0.75       341
weighted avg       0.77      0.75      0.75       341


Confusion matrix:
[[58  1  3 12]
 [ 0 72  9  0]
 [24 26 94  2]
 [ 6  0  1 33]]

Example predicted probabilities (first 5 rows):
       High       Low    Medium    Urgent
0  0.020244  0.155534  0.819366  0.004856
1  0.806716  0.000765  0.102526  0.089994
2  0.576646  0.000570  0.038367  0.384417
3  0.026291  0.366054  0.602503  0.005151
4  0.012202  0.035608  0.946403  0.005787
